# HW2 - Model Evaluation and Selection

See Canvas for details on how to complete and submit this assignment.

## Introduction

This assignment covers the second half of our regression unit (lectures 05b through Day 12). You'll move from *building* models (HW1) to *evaluating and selecting* them. The core question shifts from "what models can I create?" to "which model should I use, and how do I know?"

You'll practice cross-validation for honest performance estimates, regularization for managing model complexity, and Pipelines for safe preprocessing. You'll also read a paper that challenges assumptions about when simple models with more data beat complex models with less.

### Learning Objectives

By completing this assignment, you will demonstrate ability to:

- Articulate the Unreasonable Effectiveness of Data thesis and connect it to regularization
- Explain subset selection methods and how regularization differs fundamentally
- Implement cross-validation for model comparison and hyperparameter selection
- Apply Ridge and Lasso regression with proper feature standardization
- Build sklearn Pipelines that prevent data leakage
- Use KNN regression as a non-parametric baseline and tune k with CV
- Compare models using the baseline hierarchy (Dummy, OLS, Ridge, KNN)

### Time Estimate

This assignment should take 4-6 hours to complete.

### Generative AI Allowance

You may use GenAI tools for brainstorming, explanations, and code sketches if you disclose it, understand it, and validate it. Your submission must represent your own work and you are solely responsible for its correctness.

### Scoring

| Section | Points |
|---------|-------:|
| Reading: Unreasonable Effectiveness of Data | 15 |
| Conceptual: Feature Selection & Regularization | 15 |
| Applied: College Model Evaluation | 60 |
| Reflection | 10 |
| Total | 100 |

## Reading: The Unreasonable Effectiveness of Data

### Background

Halevy, Norvig, and Pereira's "The Unreasonable Effectiveness of Data" (2009) argues that for many problems, simple models trained on massive datasets outperform complex models trained on smaller ones. This has direct implications for how we think about regularization, model selection, and the bias-variance tradeoff.

Read the paper (available on Canvas). It's short and accessible - read the whole thing.

### Reflection (15 pts)

Write a brief reflection (2-3 concise paragraphs rich with information and personal insight) discussing how this paper relates to our study of regression and model selection. Consider:

- How the bias-variance tradeoff is illuminated by the paper's discussion of simple models with massive data
- The connection between the paper's views on memorization and our discussions of regularization/shrinkage methods
- How the paper's principles might influence your approach to choosing between complex and simple models
- The implications for cross-validation and model evaluation strategies

If you used AI tools to help understand the paper, briefly describe what you used, how you used it, and how that worked out.

---

##### UEOD Reflection

*Write your reflection here (2-3 paragraphs)...*

---

## Conceptual: Feature Selection and Regularization (15 pts)

We perform best subset, forward stepwise, and backward stepwise selection on a single data set. For each approach, we obtain $p + 1$ models, containing $0, 1, 2, \ldots, p$ predictors. Explain your answers to each of the following questions.

**(a)** Which of the three models with $k$ predictors has the smallest *training* RSS?

---

##### Answer (a)

*Your answer here...*

---

**(b)** Which of the three models with $k$ predictors has the smallest *test* RSS?

---

##### Answer (b)

*Your answer here...*

---

**(c)** True or False (with justification for each):

1. The predictors in the $k$-variable model identified by forward stepwise are a subset of the predictors in the $(k+1)$-variable model identified by forward stepwise selection.
2. The predictors in the $k$-variable model identified by backward stepwise are a subset of the predictors in the $(k+1)$-variable model identified by backward stepwise selection.
3. The predictors in the $k$-variable model identified by backward stepwise are a subset of the predictors in the $(k+1)$-variable model identified by forward stepwise selection.
4. The predictors in the $k$-variable model identified by forward stepwise are a subset of the predictors in the $(k+1)$-variable model identified by backward stepwise selection.
5. The predictors in the $k$-variable model identified by best subset are a subset of the predictors in the $(k+1)$-variable model identified by best subset selection.

---

##### Answer (c)

*Your answers here...*

---

**(d)** Feature selection methods like best subset, forward stepwise, and backward stepwise control model complexity by selecting specific predictors to include. Regularization offers an alternative approach to managing complexity. Considering Lasso regression relative to ordinary least squares:

1. Which is more flexible and why?
2. Under what conditions will Lasso provide improved prediction accuracy compared to least squares?
3. How does Lasso's approach to feature selection differ fundamentally from the three methods discussed in parts (a)-(c)?
4. How does the UEOD thesis relate to your answer in 2? That is, what does the argument that "simple models trained on massive data outperform complex models trained on less data" imply about when regularization matters most?

---

##### Answer (d)

*Your answers here...*

---

## Applied: College Model Evaluation (60 pts)

In this section you'll work with the College dataset to practice the full model evaluation workflow: standardize features, compare regularization methods, build safe Pipelines, and establish a baseline hierarchy.

Starter code is provided in some cells. It is meant to be helpful but makes assumptions about how you approach each problem, especially variable names. You are not obligated to use it, and are welcome to adapt or remove it as you see fit.

Dataset: College from Introduction to Statistical Learning (modified with engineered features)

The dataset contains information on 777 US colleges from the 1995 US News and World Report. We'll predict `Grad.Rate` (graduation rate) from 50 predictors - 17 original features plus 33 engineered polynomial and interaction terms. The large number of correlated predictors makes this an ideal setting for regularization.

### Setup

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### Load and Engineer Features

The following code loads the College dataset and generates 33 additional polynomial and interaction terms, for a total of 50 predictors. This introduces correlation between predictors (multicollinearity) that makes regularization methods shine. The process of generating new features from observed data is called feature engineering.

You are not expected to understand or recreate this code. Review it if you like, or just run it.

In [ ]:
from itertools import combinations

# download the data set directly from the web using pandas
url = "https://raw.githubusercontent.com/olearydj/INSY7120/refs/heads/main/notebooks/data/College.csv"
college = pd.read_csv(url)

# make a copy to avoid modifying the original
college_expanded = college.copy()

# set college names as index
if college_expanded.columns[0].lower() in ['name', 'college', 'school', 'institution', 'unnamed: 0']:
    college_expanded.set_index(college_expanded.columns[0], inplace=True)

# drop non-numeric 'Private' column
if 'Private' in college_expanded.columns:
    college_expanded = college_expanded.drop('Private', axis=1)

# separate response from features
response = 'Grad.Rate'
features = [col for col in college_expanded.columns if col != response]
features = [col for col in features if pd.api.types.is_numeric_dtype(college_expanded[col])]

# quadratic terms for selected features
quadratic_features = ['Apps', 'Accept', 'Enroll', 'Top10perc', 'Top25perc',
                      'F.Undergrad', 'P.Undergrad', 'Outstate']
quadratic_features = [f for f in quadratic_features if f in features]
for feature in quadratic_features:
    college_expanded[f'{feature}_sq'] = college_expanded[feature] ** 2

# add acceptance rate
if 'Accept.Rate' not in features and 'Apps' in features and 'Accept' in features:
    college_expanded['Accept.Rate'] = college_expanded['Accept'] / college_expanded['Apps']
    features.append('Accept.Rate')

# interaction terms
selectivity = [f for f in ['Accept.Rate', 'Top10perc', 'Top25perc'] if f in features]
costs = [f for f in ['Outstate', 'Room.Board', 'Expend'] if f in features]
faculty = [f for f in ['PhD', 'Terminal', 'S.F.Ratio'] if f in features]
students = [f for f in ['Top10perc', 'Top25perc', 'F.Undergrad'] if f in features]

interaction_pairs = [(s, c) for s in selectivity for c in costs]
interaction_pairs += [(f, s) for f in faculty for s in students]
interaction_pairs += [
    ('Outstate', 'perc.alumni'), ('Room.Board', 'Personal'),
    ('Apps', 'Accept.Rate'), ('S.F.Ratio', 'Expend')
]
interaction_pairs = [(a, b) for a, b in interaction_pairs
                     if a in features and b in features]
for feat1, feat2 in interaction_pairs:
    college_expanded[f'{feat1}_x_{feat2}'] = college_expanded[feat1] * college_expanded[feat2]

# log transforms for skewed features
for feature in ['Apps', 'Accept', 'Enroll']:
    if feature in features:
        college_expanded[f'log_{feature}'] = np.log1p(college_expanded[feature])

print(f"Total features: {college_expanded.shape[1] - 1}")
college_expanded.head()

### Prepare Data

In [ ]:
X = college_expanded.drop(columns=[response])
y = college_expanded[response]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set: {X_train.shape[0]} observations, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} observations")

### Standardize Features (5 pts)

Regularization methods are sensitive to feature scales. Use `StandardScaler` to standardize the features.

For now, fit the scaler on the training data and transform both train and test sets separately. Do *not* use a `Pipeline` for this task; apply `StandardScaler` manually.

In [ ]:
# Standardize features
# Your code here...

---

##### Standardization Questions

1. How can you confirm the standardization worked correctly?
2. What scale is the predictor data in after transformation?
3. Why do we fit the scaler on training data only, then transform both sets? *(The Pipeline section will show a tool that enforces this automatically.)*

*Your answers here...*

---

### Baseline: OLS Regression (5 pts)

Fit a standard `LinearRegression` on the standardized training data. Report training and test R².

In [ ]:
# Baseline OLS model
# Your code here...

---

##### OLS Interpretation

With 50 correlated predictors across 777 observations, predict how you expect OLS to perform *before* looking at your results - specifically, how will training and test R² relate to each other, and why? Then check your prediction against your output and interpret what the gap reveals.

*Your answer here...*

---

### Ridge Regression (10 pts)

Fit Ridge regression with `alpha` values of 0.1, 1.0, and 10.0. Report training and test R² for each.

Then use `RidgeCV` to find the best alpha automatically. Compare its performance with your manual choices.

In [ ]:
for alpha in [0.1, 1.0, 10.0]:
    # Fit Ridge(alpha=alpha) on X_train_scaled
    # Print training and test R²
    # Your code here...

In [ ]:
# RidgeCV for automatic alpha selection
# Your code here...

---

##### Ridge Interpretation

*Before fitting*: this dataset has 50 predictors, many of which are correlated by construction (polynomial and interaction terms derived from the same base features). Predict whether Ridge will generalize better or worse than OLS on test data, and explain your reasoning.

*After fitting:*

1. How did your prediction compare to your results? What does the gap between OLS and Ridge test R² reveal about this dataset?
2. How does performance change across your three manual alpha values? Were your choices a good range to search?
3. What does the *magnitude* of RidgeCV's selected alpha tell you about how strongly regularization is needed here?

*Your answer here...*

---

### Lasso Regression (10 pts)

Fit Lasso regression with `alpha` values of 0.1, 1.0, and 10.0. Report training and test R² and the number of features retained (non-zero coefficients) for each.

Then use `LassoCV` to find the best alpha automatically.

In [ ]:
for alpha in [0.1, 1.0, 10.0]:
    # Fit Lasso(alpha=alpha, max_iter=10000) on X_train_scaled
    # Count non-zero coefficients: np.sum(model.coef_ != 0)
    # Print training R², test R², and feature count
    # Your code here...

In [ ]:
# LassoCV for automatic alpha selection
# Your code here...

---

##### Lasso Interpretation

1. How does the number of retained features change as alpha increases from 0.1 to 10.0? Connect this pattern to the bias-variance tradeoff: what are you trading away at each extreme?
2. LassoCV identified an optimal alpha that retains only a subset of features. A classmate argues: "Since Lasso dropped those features, they must be useless - we should remove them permanently and refit OLS on only the retained features." Identify the flaw in this reasoning.
3. Compare Lasso to Ridge in terms of test R² and number of features used. If Lasso achieves similar performance with far fewer features, what does this suggest about the structure of the predictor set?

*Your answer here...*

---

### Compare Coefficients (5 pts)

Create a comparison of coefficients between OLS, Ridge (best alpha), and Lasso (best alpha). Consider which features each method considers most important, how many features Lasso zeros out, and whether the methods agree on the top predictors.

In [ ]:
# Refit with best alphas from CV
best_ridge = Ridge(alpha=best_ridge_alpha)
best_ridge.fit(X_train_scaled, y_train)

best_lasso = Lasso(alpha=best_lasso_alpha, max_iter=10000)
best_lasso.fit(X_train_scaled, y_train)

coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'OLS':     # your code here,
    'Ridge':   # your code here,
    'Lasso':   # your code here,
})

# Sort by absolute OLS coefficient and display the top 15 rows
# Your code here...

---

##### Coefficient Comparison

What does this comparison tell you about the different ways Ridge and Lasso perform regularization? Connect your observations to the "OLS breaks, Ridge shares, Lasso chooses" mental model from lecture.

*Your answer here...*

---

### Pipeline (10 pts)

In the Standardize Features section, you manually fit `StandardScaler` on `X_train` and applied it to both `X_train` and `X_test` before fitting Ridge. That approach works, but requires careful bookkeeping: any mistake in that sequence silently introduces leakage.

`Pipeline` chains preprocessing and modeling into a single estimator. The key payoff: when `cross_val_score` evaluates a Pipeline, it re-fits the scaler on each fold's training data and transforms the test fold without refitting - leakage is structurally impossible.

Build a `Pipeline` with `StandardScaler` and `Ridge` using the best alpha from `RidgeCV`. Fit on the raw, unscaled `X_train` - the Pipeline handles scaling internally.

In [ ]:
# Build and fit a Pipeline: StandardScaler → Ridge (best alpha from RidgeCV)
# Your code here...

Report training and test R² for your Pipeline. Compare them to your manually-scaled Ridge results.

In [ ]:
# Pipeline training and test R²
# Your code here...

Use `cross_val_score` with your Pipeline to get honest CV performance estimates on the training set.

In [ ]:
# 5-fold CV with Pipeline (scoring='r2')
# Your code here...

---

##### Pipeline Questions

1. How do the Pipeline train/test R² compare to your manually-scaled Ridge results? Are they the same or different? Explain why.
2. Why is it important to pass raw, unscaled training data to the Pipeline rather than pre-scaled data? What does the Pipeline do internally that you had to do manually in the Standardize Features section?
3. `cross_val_score` re-fits the entire Pipeline on each fold's training data. Why is this safe from leakage when the same call with a pre-scaled dataset and a bare `Ridge` model would not be?

*Your answers here...*

---

### KNN Regression (10 pts)

KNN regression predicts by averaging the target values of the k nearest training points. It makes no assumptions about the functional form of the relationship, making it a useful non-parametric complement to the linear models you've already fit.

KNN uses Euclidean distance, so features on different scales dominate the calculation. Use a `Pipeline` with `StandardScaler` for all KNN models in this section - pass raw `X_train` and `X_test` to each Pipeline.

Fit KNN Pipelines for k ∈ {1, 3, 5, 10, 20, 50}. Report training and test R² for each k.

In [ ]:
k_values = [1, 3, 5, 10, 20, 50]

# For each k, build Pipeline(StandardScaler, KNeighborsRegressor(n_neighbors=k))
# Fit on X_train, score on both X_train and X_test
# Collect results in a DataFrame
# Your code here...

Use `cross_val_score` to evaluate each k on the training set.

In [ ]:
cv_r2_list = []
for k in k_values:
    # Build a Pipeline for this k and compute 5-fold CV R²
    # Append the mean CV R² to cv_r2_list
    # Your code here...

knn_df['cv_r2'] = cv_r2_list
best_k = k_values[int(np.argmax(cv_r2_list))]
print(f"Best k: {best_k}")
print(knn_df.to_string(index=False))

Plot CV R² vs. k, then identify the best k. The plot code below is complete and correct as written, with assumptions about your results and variable names.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, knn_df['train_r2'], 'o-',  label='Training R²')
ax.plot(k_values, knn_df['test_r2'],  's-',  label='Test R²')
ax.plot(k_values, knn_df['cv_r2'],    '^--', label='CV R²')
ax.set_xlabel('k (number of neighbors)')
ax.set_ylabel('R²')
ax.set_title('KNN Performance vs. k')
ax.legend()
plt.tight_layout()
plt.show()

Fit a final Pipeline with the best k and report its training and test R².

In [ ]:
# Final KNN Pipeline with best k
# Your code here...

---

##### KNN Interpretation

1. *Before running your k-loop*: predict the shape of both the training R² and test R² curves as k increases from 1 to 50. Then run your code. Did the actual curves match your prediction? Describe the patterns and explain how they reflect the bias-variance tradeoff.
2. Why must KNN use feature scaling? What would happen to distance calculations if predictors were on different scales?
3. How does the best KNN model compare to Ridge in terms of test R²? What does this gap (or lack of one) suggest about the nature of the relationship between predictors and graduation rate?
4. This dataset has 50 predictors. If you increased that to 500 while keeping n=777 fixed, how would you expect KNN's performance to change relative to Ridge, and why? What property of KNN drives this sensitivity?

*Your answers here...*

---

### Model Summary (5 pts)

Create a summary table comparing all models: Dummy (baseline), OLS, Ridge (best alpha), Lasso (best alpha), and KNN (best k). Include training R², test R², CV R² (mean ± std), and number of features used.

Compute CV R² for all models here using `cross_val_score` on `X_train` — even for models fit earlier in the notebook. Use Pipeline-wrapped versions so the scaler is re-fit correctly on each fold.

In [ ]:
models = {
    'Dummy': Pipeline([('scaler', StandardScaler()), ('model', DummyRegressor(strategy='mean'))]),
    'OLS':   Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    # Add Ridge, Lasso, and KNN entries using best_ridge_alpha, best_lasso_alpha, best_k
    # Your code here...
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    rows.append({
        'Model':      name,
        'Train R²':   model.score(X_train, y_train),
        'Test R²':    model.score(X_test,  y_test),
        'CV R² mean': cv.mean(),
        'CV R² std':  cv.std(),
        # Features: number of non-zero coefficients
        # Hint: model.named_steps['model'] accesses the fitted estimator inside the Pipeline
        'Features':   0  # replace with your code
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

---

##### Overall Analysis

Based on your results:

1. Which model would you recommend for predicting college graduation rates? Justify considering both performance and complexity.
2. Interpret each gap in your model comparison table:
   - OLS → Ridge: does regularization help? What does this tell you about the predictors?
   - Ridge → Lasso: does sparsity help? What does this tell you about feature redundancy?
   - Best linear model → KNN: is there non-linearity worth pursuing?

   What overall picture of the College dataset emerges from reading these gaps together?
3. How do your results connect to the Unreasonable Effectiveness of Data paper? Does this dataset favor simple or complex models?

*Your analysis here...*

---

## Reflection (10 pts)

Address the following (concise bullets or short paragraphs are fine):

### 1. Key Takeaway

What part of this assignment most improved your understanding of model evaluation or selection? Include a concrete example of something you understand better now than before.

---

##### Key Takeaway

*Your response here...*

---

### 2. GenAI Use

If you used GenAI tools (ChatGPT, Claude, etc.):
- What tool/model did you use?
- How did you use it (understanding concepts, debugging code, etc.)?
- How did you verify the output was correct?
- What worked well and what didn't?

If you didn't use GenAI, explain why and whether you plan to use it for future assignments.

---

##### GenAI Use

*Your response here...*

---

### 3. Feedback

- Approximately how much time did you spend on this assignment?
- What was the most difficult part?
- How would you improve this assignment?
- Anything else you want to share or ask?

---

##### Feedback

*Your response here...*

---